# EDA do atraso de entrega e do risco de SLA no Olist

## O problema de negócio

Em um marketplace, a experiência do cliente não termina na confirmação da compra: ela inclui receber o pedido dentro do prazo prometido. Um atraso pode aumentar contatos no atendimento, reclamações, cancelamentos e perda de confiança. Para a operação, também pode sinalizar dificuldades de vendedores, transportadoras, rotas, regiões ou categorias de produto.

Este projeto busca entender quando o processo de entrega ultrapassa a promessa feita ao cliente e quais padrões históricos ajudam a explicar esse risco. A EDA deve produzir evidências para uma etapa posterior de modelagem: uma regressão para o atraso em dias e uma classificação para a ocorrência de violação.

SLA (*Service Level Agreement*) é um compromisso mensurável de nível de serviço. Aqui, o compromisso analisado é o prazo de entrega apresentado ao cliente: `order_estimated_delivery_date`. O prazo prometido é a diferença entre essa data estimada e a data da compra.

O dataset não traz uma coluna chamada SLA nem o contrato comercial completo da Olist. Por isso, adota-se uma definição operacional: há **violação de SLA** quando `order_delivered_customer_date` é posterior a `order_estimated_delivery_date`; o atraso é a diferença entre essas duas datas. Essa é uma proxy analítica do cumprimento do prazo prometido, e não uma afirmação sobre todos os contratos reais da Olist.

Ao final desta etapa, deve estar disponível uma leitura auditável do processo de entrega: (1) uma população supervisionada e uma tabela analítica com uma linha por pedido; (2) evidências sobre distribuição do atraso, períodos, destinos, vendedores e categorias associados a maior risco histórico; (3) contratos explícitos sobre disponibilidade temporal e qualidade dos dados; e (4) uma base preparada para comparar regressão e classificação sem usar informação posterior à compra.

O prazo analisado é o prazo de cada pedido, não um prazo externo do projeto: ele vai da compra até a data estimada de entrega ao cliente. A EDA também deve deixar claro onde o dataset não permite responder às perguntas de negócio.

## Escopo desta EDA

O resultado diretamente observável é o cumprimento do prazo no nível do pedido. Distância vendedor–cliente, composição de vendedores, categoria e indicadores históricos de despacho serão tratados como variáveis explicativas ou proxies. Não há dados suficientes para medir entrega parcial, rota real, hub do transportador ou SLA individual por vendedor.

## População e medidas de referência

A análise começa com o inventário da fonte, a população supervisionada e as medidas de atraso e de prazo. Essa etapa estabelece o denominador que será reutilizado nas análises temporal, geográfica e por categoria.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

LANG = "pt"
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ecom_delivery_sla import (
    anomaly_summary,
    build_supervised_orders,
    category_summary,
    inventory_summary,
    lead_summary,
    monthly_summary,
    plot_category_breach,
    plot_delay_distribution,
    plot_lead_scatter,
    plot_monthly_breach,
    plot_state_breach,
    read_olist_tables,
    state_summary,
    target_summary,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'olist' / 'csv'
tables = read_olist_tables(DATA_DIR)
supervised = build_supervised_orders(tables['orders'])
display(inventory_summary(tables, language=LANG))
display(target_summary(supervised, language=LANG))
display(lead_summary(supervised, language=LANG))

,tabela,linhas,colunas
0,pedidos,99441,8
1,clientes,99441,5
2,itens,112650,7
3,produtos,32951,9


,métrica,valor
0,Pedidos supervisionáveis,96470.000000
1,Violações de SLA,6534.000000
2,Taxa de violação,0.067731
3,Mediana do atraso (dias),-11.948102
4,P95 do atraso (dias),3.816993
5,P99 do atraso (dias),18.939553
6,Atraso > 30 dias,360.000000
7,Antecipação < -30 dias,2352.000000


,medida,mediana_dias,p95_dias
0,Prazo prometido,23.230880,38.418493
1,Prazo realizado,10.217477,29.274046


### Insight da população

A análise parte de 96.470 pedidos entregues com datas suficientes para o alvo. Foram observadas 6.534 violações pela regra de dia de calendário, ou 6,77% da população supervisionada. O prazo prometido tem mediana de 23,23 dias, enquanto o prazo real tem mediana de 10,22 dias; essa diferença não deve ser lida como desempenho futuro, pois o prazo real só é conhecido depois da entrega.

## Distribuição do atraso e relação com o prazo prometido

O primeiro gráfico mostra a distribuição do atraso em dias, com zero como fronteira entre entrega antecipada e violação. O segundo compara o prazo prometido com o prazo real. A comparação é descritiva: o prazo real é uma medida pós-evento e não pode entrar como variável preditora de um modelo operacional.

In [2]:
display(plot_delay_distribution(supervised, language=LANG))
display(plot_lead_scatter(supervised, language=LANG))

### Insight da distribuição

A mediana do atraso é de aproximadamente −11,95 dias (linha verde), portanto, a maior concentração à esquerda do zero indica que a maioria dos pedidos foi entregue antes do prazo prometido. Ainda assim, a cauda de atrasos é relevante: o percentil 95 é 3,82 dias e o máximo chega a cerca de 188,98 dias. A linha vermelha em zero separa entregas antecipadas das entregas atrasadas. A correlação descritiva entre atraso e prazo prometido é −0,50; isso sugere que a promessa está associada ao desfecho, mas não identifica causalidade nem autoriza usar informação pós-entrega.

## Variação temporal

A tabela e o gráfico seguintes calculam a taxa mensal de violação entre os pedidos supervisionados. O objetivo é verificar se o risco observado é estável ou se existem períodos que exigem validação temporal e investigação operacional. Meses com poucos pedidos devem ser interpretados com cautela.

In [3]:
display(monthly_summary(supervised, language=LANG))
display(plot_monthly_breach(supervised, language=LANG))

,pedidos,violacoes,atraso_mediano,atraso_p95,taxa_violacao
mês_da_compra,,,,,
2016-09,1,1,36.324745,36.324745,1.000000
2016-10,265,2,-38.744630,-10.388572,0.007547
2016-12,1,0,-21.336991,-21.336991,0.000000
2017-01,750,22,-28.329508,-9.332899,0.029333
2017-02,1653,49,-19.412593,-4.417368,0.029643
2017-03,2546,116,-13.315289,0.590298,0.045562
2017-04,2303,151,-13.480567,2.636439,0.065567
2017-05,3545,106,-13.428843,-1.623907,0.029901
2017-06,3135,95,-12.422407,-1.394035,0.030303


### Insight temporal

A taxa não é constante: os picos mais claros aparecem em novembro de 2017 (14,31%), fevereiro de 2018 (15,99%) e março de 2018 (21,36%). Em abril e junho de 2018, as taxas caem para 5,31% e 1,36%, respectivamente. Esses contrastes justificam uma divisão temporal e análises de estabilidade, mas não provam que um mês específico causou o atraso.

## Variação por estado do cliente

Agora é comparada a taxa de violação por estado de destino. Mantêm-se apenas grupos com pelo menos 500 pedidos para reduzir a instabilidade de proporções pequenas. O estado é um marcador geográfico descritivo; ele pode refletir composição de vendedores, distância, categoria e período.

In [4]:
display(state_summary(supervised, tables['customers'], language=LANG, min_orders=500))
display(plot_state_breach(supervised, tables['customers'], language=LANG, min_orders=500))

,pedidos,violacoes,atraso_mediano,taxa_violacao
estado_cliente,,,,
PR,4923,199,-13.093970,0.040423
SP,40494,1820,-10.282726,0.044945
MG,11354,519,-13.118380,0.045711
DF,2080,118,-12.064184,0.056731
MT,886,53,-14.263432,0.059819
RS,5344,325,-14.028160,0.060816
GO,1957,128,-12.305347,0.065406
SC,3546,291,-12.082899,0.082064
PE,1593,153,-13.584977,0.096045


### Insight geográfico

Entre os estados com volume mínimo, Maranhão apresenta a maior taxa observada (19,67%), seguido por Ceará (15,32%), Bahia (14,04%) e Rio de Janeiro (13,47%). Paraná (5,00%), Minas Gerais (5,61%) e São Paulo (5,89%) aparecem entre as menores. A diferença é útil para orientar hipóteses e estratificação, mas não deve ser interpretada como efeito causal do estado.

## Categorias e auditoria temporal

A tabela de itens pode relacionar vários produtos ao mesmo pedido. Para comparar os pedidos por categoria, cada pedido recebe provisoriamente a categoria que aparece mais vezes entre os seus itens, chamada de **categoria predominante**. Essa é uma simplificação para a EDA, e a função aplica um mínimo de 1.000 pedidos por grupo. Em paralelo, a auditoria verifica anomalias temporais sem corrigi-las silenciosamente.

In [5]:
display(category_summary(supervised, tables['items'], tables['products'], language=LANG, min_orders=1000))
display(plot_category_breach(supervised, tables['items'], tables['products'], language=LANG, min_orders=1000, top_n=12))
display(anomaly_summary(supervised, language=LANG))

,pedidos,violacoes,atraso_mediano,taxa_violacao
categoria_modal,,,,
moveis_escritorio,1244,101,-12.192078,0.081190
bebes,2794,226,-11.240376,0.080888
eletronicos,2509,192,-11.118102,0.076525
beleza_saude,8636,649,-12.038576,0.075151
cama_mesa_banho,9228,689,-11.281481,0.074664
relogios_presentes,5456,406,-12.048600,0.074413
automotivo,3803,278,-11.285660,0.073100
moveis_decoracao,6182,446,-11.624120,0.072145
telefonia,4069,291,-11.101296,0.071516


,regra,casos
0,Entrega ao cliente antes do despacho,23
1,Despacho ausente,1
2,Aprovação antes da compra,0
3,Entrega ao cliente antes da compra,0


### Insight de categoria e qualidade temporal

Usando essa aproximação de categoria predominante, eletrônicos (9,84%), móveis de escritório (9,24%) e bebês (9,23%) ficam acima da taxa global, enquanto malas e acessórios (5,45%), pet shop (6,31%) e moda/bolsas/acessórios (6,67%) ficam abaixo. Isso não significa que todos os itens de um pedido pertençam à mesma categoria; significa apenas que foi escolhida a categoria mais frequente para obter uma primeira comparação. A auditoria encontrou 23 entregas reais anteriores ao despacho do transportador e um despacho sem data; não foram encontrados casos de entrega anterior à compra. Esses registros precisam de uma decisão de qualidade de dados antes da modelagem.

## Conclusão e próximo passo

O alvo é calculável, desbalanceado e varia no tempo, por destino e por categoria. A EDA sustenta a continuidade do projeto, mas os achados ainda são descritivos. O próximo passo é construir a tabela analítica com uma linha por pedido, definir o corte temporal, explicitar a política para múltiplos vendedores e separar variáveis disponíveis na compra de variáveis pós-entrega.

Até a aprovação dos contratos, `production_eligible = false`.